
# 02 Cleaning: ETL Pipeline & Data Standardisation (Reset)

**Goal**
Build a transparent, production-style cleaning pipeline that transforms raw VC data into a standardised analytical dataset.

**Deliverables from this notebook**
- `data/processed/startups_cleaned.csv`
- `data/processed/startups_cleaned.parquet`
- `data/processed/etl_run_log.csv`

**Methodology**
Every transformation is logged with: step name, rationale, rows/columns before-after, and records affected.


In [1]:

from pathlib import Path
from datetime import datetime
import re
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)


In [2]:

PROJECT_ROOT = Path.cwd().resolve().parent
RAW_PATH = PROJECT_ROOT / 'data' / 'raw' / 'investments_VC.csv'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print('Raw path      :', RAW_PATH)
print('Processed path:', PROCESSED_DIR)


Raw path      : /Users/rashmianand/Desktop/SectionC_G17_WhyStartupsFail/data/raw/investments_VC.csv
Processed path: /Users/rashmianand/Desktop/SectionC_G17_WhyStartupsFail/data/processed


In [3]:

transformation_log = []

def log_step(step_name, why, before_df, after_df, affected_rows=None, notes=''):
    transformation_log.append({
        'timestamp': datetime.now().isoformat(timespec='seconds'),
        'step': step_name,
        'why': why,
        'rows_before': int(before_df.shape[0]),
        'rows_after': int(after_df.shape[0]),
        'cols_before': int(before_df.shape[1]),
        'cols_after': int(after_df.shape[1]),
        'records_affected': None if affected_rows is None else int(affected_rows),
        'notes': notes
    })

def parse_money(value):
    """
    WHY: Raw monetary fields mix formats like '17,50,000', '40,000', '$1,200,000', and blanks.
    This parser keeps only numeric signal and converts safely to float.
    """
    if pd.isna(value):
        return np.nan
    if isinstance(value, (int, float, np.number)):
        return float(value)

    s = str(value).strip()
    if s == '':
        return np.nan

    # Remove currency symbols/text, keep digits, dot, comma, minus
    s = re.sub(r'[^0-9,.-]', '', s)
    if s in {'', '-', '.', ','}:
        return np.nan

    # Handle mixed comma formats by removing commas entirely
    s = s.replace(',', '')
    try:
        return float(s)
    except ValueError:
        return np.nan



## Step 1: Load Raw Dataset
**Why this matters**
- Establishes a controlled start point for reproducible cleaning.
- Uses `latin1` to prevent decode failures from non-UTF-8 characters.


In [4]:

df_raw = pd.read_csv(RAW_PATH, encoding='latin1')
df = df_raw.copy()

log_step(
    step_name='load_raw_data',
    why='Load immutable raw dataset with compatible encoding.',
    before_df=df_raw,
    after_df=df
)

print('Raw shape:', df.shape)
df.head(3)


Raw shape: (54294, 39)


,permalink,name,homepage_url,category_list,market,funding_total_usd,status,country_code,state_code,region,city,funding_rounds,founded_at,founded_month,founded_quarter,founded_year,first_funding_at,last_funding_at,seed,venture,equity_crowdfunding,undisclosed,convertible_note,debt_financing,angel,grant,private_equity,post_ipo_equity,post_ipo_debt,secondary_market,product_crowdfunding,round_A,round_B,round_C,round_D,round_E,round_F,round_G,round_H
0,/organization/waywire,#waywire,http://www.waywire.com,|Entertainment|Politics|Social Media|News|,News,"17,50,000",acquired,USA,NY,New York City,New York,1.0,2012-06-01,2012-06,2012-Q2,2012.0,2012-06-30,2012-06-30,1750000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,/organization/tv-communications,&TV Communications,http://enjoyandtv.com,|Games|,Games,"40,00,000",operating,USA,CA,Los Angeles,Los Angeles,2.0,NaN,NaN,NaN,NaN,2010-06-04,2010-09-23,0.0,4000000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,/organization/rock-your-paper,'Rock' Your Paper,http://www.rockyourpaper.org,|Publishing|Education|,Publishing,"40,000",operating,EST,NaN,Tallinn,Tallinn,1.0,2012-10-26,2012-10,2012-Q4,2012.0,2012-08-09,2012-08-09,40000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0



## Step 2: Standardize Column Names
**Why this matters**
- The raw file contains spacing inconsistencies (e.g., `' market '` and `' funding_total_usd '`).
- Standardized names reduce downstream errors and improve code readability.


In [5]:

before = df.copy()
old_cols = df.columns.tolist()

df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(r'[^a-z0-9]+', '_', regex=True)
      .str.strip('_')
)

changed = sum(1 for a, b in zip(old_cols, df.columns) if a != b)

log_step(
    step_name='standardize_column_names',
    why='Normalize naming conventions and remove hidden whitespace/symbol variance.',
    before_df=before,
    after_df=df,
    affected_rows=changed,
    notes='records_affected = number of columns renamed'
)

df.columns.tolist()


['permalink',
 'name',
 'homepage_url',
 'category_list',
 'market',
 'funding_total_usd',
 'status',
 'country_code',
 'state_code',
 'region',
 'city',
 'funding_rounds',
 'founded_at',
 'founded_month',
 'founded_quarter',
 'founded_year',
 'first_funding_at',
 'last_funding_at',
 'seed',
 'venture',
 'equity_crowdfunding',
 'undisclosed',
 'convertible_note',
 'debt_financing',
 'angel',
 'grant',
 'private_equity',
 'post_ipo_equity',
 'post_ipo_debt',
 'secondary_market',
 'product_crowdfunding',
 'round_a',
 'round_b',
 'round_c',
 'round_d',
 'round_e',
 'round_f',
 'round_g',
 'round_h']


## Step 3: Clean Text Fields and Canonical Missing Values
**Why this matters**
- Free-text fields often contain leading/trailing spaces and placeholder tokens.
- Converting blank-like values to `NaN` creates a single missing-data standard.


In [6]:

before = df.copy()

text_cols = df.select_dtypes(include='object').columns
for c in text_cols:
    df[c] = df[c].astype(str).str.strip()
    df[c] = df[c].replace({'': np.nan, 'nan': np.nan, 'None': np.nan, 'NULL': np.nan, 'N/A': np.nan})

# Status/category/market normalization for consistency
for c in ['status', 'market', 'country_code', 'state_code', 'region', 'city', 'category_list']:
    if c in df.columns:
        df[c] = df[c].astype('string').str.strip()

if 'status' in df.columns:
    df['status'] = df['status'].str.lower()

affected = int((before[text_cols].fillna('') != df[text_cols].fillna('')).sum().sum())

log_step(
    step_name='clean_text_and_missing_tokens',
    why='Trim text noise and convert placeholder-like values into true missing markers.',
    before_df=before,
    after_df=df,
    affected_rows=affected
)


/var/folders/0x/qxs60c856d3dnd91shmgplfw0000gn/T/ipykernel_89327/1369998911.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_cols = df.select_dtypes(include='object').columns



## Step 4: Remove Duplicates
**Why this matters**
- Duplicate records inflate counts and bias KPI/statistical outcomes.
- We remove exact row duplicates first, then duplicate entities by `permalink`.


In [7]:

before = df.copy()

exact_dup = int(df.duplicated().sum())
df = df.drop_duplicates().copy()

permalink_dup = 0
if 'permalink' in df.columns:
    permalink_dup = int(df.duplicated(subset=['permalink']).sum())
    df = df.drop_duplicates(subset=['permalink'], keep='first').copy()

log_step(
    step_name='remove_duplicates',
    why='Prevent duplicate-company overcounting in downstream analyses.',
    before_df=before,
    after_df=df,
    affected_rows=exact_dup + permalink_dup,
    notes=f'exact_duplicates={exact_dup}; permalink_duplicates={permalink_dup}'
)



## Step 5: Parse and Validate Dates
**Why this matters**
- Date fields are needed for trend, survival, and cohort analyses.
- Invalid dates must be safely coerced to missing values.


In [8]:

before = df.copy()

date_cols = [
    'founded_at', 'first_funding_at', 'last_funding_at'
]
for c in date_cols:
    if c in df.columns:
        df[c] = pd.to_datetime(df[c], errors='coerce')

if 'founded_year' in df.columns:
    df['founded_year'] = pd.to_numeric(df['founded_year'], errors='coerce')
    current_year = datetime.now().year
    df.loc[(df['founded_year'] < 1900) | (df['founded_year'] > current_year), 'founded_year'] = np.nan
    df['founded_year'] = df['founded_year'].astype('Int64')

if 'founded_month' in df.columns:
    df['founded_month'] = df['founded_month'].astype('string').str.strip()

if 'founded_quarter' in df.columns:
    df['founded_quarter'] = df['founded_quarter'].astype('string').str.strip()

affected = 0
for c in date_cols:
    if c in before.columns:
        affected += int((before[c].astype('string') != df[c].astype('string')).sum())

log_step(
    step_name='parse_validate_dates',
    why='Ensure temporal fields are analytically valid and typed for time-series work.',
    before_df=before,
    after_df=df,
    affected_rows=affected
)



## Step 6: Standardize Numeric Funding Fields
**Why this matters**
- Funding metrics are central to startup success/failure analysis.
- Mixed numeric formatting must be normalized before any statistical work.


In [9]:

before = df.copy()

numeric_candidates = [
    'funding_total_usd', 'funding_rounds', 'seed', 'venture', 'equity_crowdfunding',
    'undisclosed', 'convertible_note', 'debt_financing', 'angel', 'grant',
    'private_equity', 'post_ipo_equity', 'post_ipo_debt', 'secondary_market',
    'product_crowdfunding', 'round_a', 'round_b', 'round_c', 'round_d',
    'round_e', 'round_f', 'round_g', 'round_h'
]

changed_cells = 0
for c in numeric_candidates:
    if c in df.columns:
        before_col = df[c].copy()
        df[c] = df[c].apply(parse_money)
        changed_cells += int((before_col.astype('string') != df[c].astype('string')).sum())

# Domain rule: negative funding values are invalid -> set to missing
for c in numeric_candidates:
    if c in df.columns:
        neg_count = int((df[c] < 0).sum())
        if neg_count > 0:
            df.loc[df[c] < 0, c] = np.nan

# Operational assumption for stage-level funding columns:
# missing often means 'not raised/disclosed' -> impute as 0 for additive analyses
stage_cols = [c for c in numeric_candidates if c in df.columns and c not in ['funding_total_usd', 'funding_rounds']]
for c in stage_cols:
    df[c] = df[c].fillna(0.0)

log_step(
    step_name='standardize_numeric_funding',
    why='Convert messy currency/number strings to numeric and enforce valid funding domain constraints.',
    before_df=before,
    after_df=df,
    affected_rows=changed_cells
)



## Step 7: Business Rule Features for Analysis Readiness
**Why this matters**
- Derived features simplify EDA and model prep.
- Explicitly tagging lifecycle outcomes improves consistency across analyses.


In [10]:

before = df.copy()

if 'status' in df.columns:
    active_states = {'operating'}
    terminal_states = {'closed', 'acquired', 'ipo'}

    df['status_group'] = np.where(
        df['status'].isin(active_states), 'active',
        np.where(df['status'].isin(terminal_states), 'terminal', 'other')
    )

    df['is_closed'] = (df['status'] == 'closed').astype('Int64')

if {'first_funding_at', 'last_funding_at'}.issubset(df.columns):
    df['funding_duration_days'] = (pd.to_datetime(df['last_funding_at'], errors='coerce') - pd.to_datetime(df['first_funding_at'], errors='coerce')).dt.days

affected = int((before.shape[0]))

log_step(
    step_name='derive_analysis_features',
    why='Create stable analytical features for status-based and timeline-based comparisons.',
    before_df=before,
    after_df=df,
    affected_rows=affected
)



## Step 8: Final Data Quality Check
**Why this matters**
- Confirms the cleaned output is coherent before persistence.
- Makes quality assumptions explicit for reproducibility.


In [11]:

quality_summary = pd.DataFrame({
    'column': df.columns,
    'dtype': [str(df[c].dtype) for c in df.columns],
    'missing_count': [int(df[c].isna().sum()) for c in df.columns],
    'missing_pct': [round(float(df[c].isna().mean() * 100), 2) for c in df.columns]
}).sort_values('missing_pct', ascending=False)

print('Final cleaned shape:', df.shape)
quality_summary.head(15)


Final cleaned shape: (49437, 42)


,column,dtype,missing_count,missing_pct
8,state_code,string,19278,39.00
13,founded_month,string,10957,22.16
15,founded_year,Int64,10957,22.16
14,founded_quarter,string,10957,22.16
12,founded_at,datetime64[us],10885,22.02
5,funding_total_usd,float64,8532,17.26
10,city,string,6117,12.37
7,country_code,string,5274,10.67
9,region,string,5274,10.67
4,market,string,3968,8.03



## Step 9: Save Processed Outputs
**Why this matters**
- Ensures downstream notebooks and BI tools consume one standardized source.
- Exports both CSV (human-readable) and Parquet (efficient analytics format).


In [12]:

cleaned_csv = PROCESSED_DIR / 'startups_cleaned.csv'
cleaned_parquet = PROCESSED_DIR / 'startups_cleaned.parquet'
etl_log_csv = PROCESSED_DIR / 'etl_run_log.csv'

df.to_csv(cleaned_csv, index=False)
df.to_parquet(cleaned_parquet, index=False)

etl_log_df = pd.DataFrame(transformation_log)
etl_log_df.to_csv(etl_log_csv, index=False)

print('Saved:', cleaned_csv)
print('Saved:', cleaned_parquet)
print('Saved:', etl_log_csv)

etl_log_df


Saved: /Users/rashmianand/Desktop/SectionC_G17_WhyStartupsFail/data/processed/startups_cleaned.csv
Saved: /Users/rashmianand/Desktop/SectionC_G17_WhyStartupsFail/data/processed/startups_cleaned.parquet
Saved: /Users/rashmianand/Desktop/SectionC_G17_WhyStartupsFail/data/processed/etl_run_log.csv


,timestamp,step,why,rows_before,rows_after,cols_before,cols_after,records_affected,notes
0,2026-04-28T15:32:22,load_raw_data,Load immutable raw dataset with compatible enc...,54294,54294,39,39,NaN,
1,2026-04-28T15:32:22,standardize_column_names,Normalize naming conventions and remove hidden...,54294,54294,39,39,10.0,records_affected = number of columns renamed
2,2026-04-28T15:32:22,clean_text_and_missing_tokens,Trim text noise and convert placeholder-like v...,54294,54294,39,39,94909.0,
3,2026-04-28T15:32:22,remove_duplicates,Prevent duplicate-company overcounting in down...,54294,49437,39,39,4857.0,exact_duplicates=4855; permalink_duplicates=2
4,2026-04-28T15:32:22,parse_validate_dates,Ensure temporal fields are analytically valid ...,49437,49437,39,39,16.0,
5,2026-04-28T15:32:22,standardize_numeric_funding,Convert messy currency/number strings to numer...,49437,49437,39,39,40905.0,
6,2026-04-28T15:32:22,derive_analysis_features,Create stable analytical features for status-b...,49437,49437,39,42,49437.0,



## Cleaning Completion Summary
- Raw data remained untouched in `data/raw/`.
- Comprehensive cleaning and standardisation steps were applied.
- Every transformation was documented with purpose (`why`) and impact.
- Cleaned dataset is now available in `data/processed/` and ready for analysis.
